In [ ]:
# Import libararies
from google.cloud import bigquery, storage
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Data Preprocessing

In [ ]:
# 1. Clean Calendar
print('Processing calendar...')
df_cal = pd.read_csv('calendar.csv', dtype={'listing_id': str})
df_cal['listing_id'] = df_cal['listing_id'].astype(str).str.strip()
df_cal.to_csv('calendar_clean.csv', index=False)

# 2. Clean Listings
print("Processing listings...")
df_list = pd.read_csv('listings.csv', encoding='latin1', low_memory=False, dtype={'id': str})
df_list['id'] = df_list['id'].astype(str).str.strip()

# Clean price column
df_list['price'] = (
    df_list['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)
df_list['price'] = pd.to_numeric(df_list['price'], errors='coerce')


# Keep essential columns
cols_to_keep = [
    'id', 'name', 'neighbourhood_cleansed', 'room_type',
    'accommodates', 'bedrooms', 'beds', 'price',
    'review_scores_rating', 'number_of_reviews', 'host_is_superhost'
]
df_list = df_list[[c for c in cols_to_keep if c in df_list.columns]]
df_list.to_csv('listings_clean.csv', index=False)

print("Both cleaned files saved successfully!")

Processing calendar...
Processing listings...
Both cleaned files saved successfully!


# Data Audit & Profiling

In [ ]:
# Define project and bucket parameters
PROJECT_ID = 'airbnb-analysis-513507'
BUCKET_NAME = 'air-bnb'

# 1. Load data directly from Google Cloud Storage
listings_uri = f'gs://{BUCKET_NAME}/listings_clean.csv'
calendar_uri = f'gs://{BUCKET_NAME}/calendar_clean.csv'

print('Loading datasets from Google Cloud Storage...')

df_listings = pd.read_csv(
    listings_uri, encoding='latin1', dtype={'id':str, 'listing_id':str}
)
df_calendar = pd.read_csv(
    calendar_uri, encoding='latin1', dtype={'listing_id':str, 'available':str}
)

print(
    f'Listings Shape: {df_listings.shape} | Calendar Shape:'
    f' {df_calendar.shape}'
)

print('=' * 60)

# 2. Inspect Duplicate Primary Keys
id_col = 'id' if 'id' in df_listings.columns else 'listing_id'
raw_listings_count = len(df_listings)
unique_listings_count = df_listings[id_col].nunique()
duplicate_count = df_listings[id_col].duplicated().sum()

print('--- LISTINGS PRIMARY KEY AUDIT ---')
print(f'Total Raw Rows: {raw_listings_count}')
print(f'Unique Listing IDs: {unique_listings_count}')
print(f'Duplicate Listing Rows: {duplicate_count}')

if duplicate_count > 0:
  print('\nTop 5 duplicated listing IDs:')
  print(df_listings[id_col].value_counts().head(5))

print('=' * 60)


# 3. Inspect Missing/Null Values in Key Analytic Fields
print('--- NULL VALUES IN CORE COLUMNS ---')
check_cols = [
    id_col,
    'name',
    'neighbourhood_cleansed',
    'room_type',
    'accommodates',
    'bedrooms',
    'beds',
    'price',
    'review_scores_rating',
    'number_of_reviews',
    'host_is_superhost',
]

existing_cols = [c for c in check_cols if c in df_listings.columns]
print(df_listings[existing_cols].isnull().sum())

print('=' * 60)


# 4. Calendar Availability Audit
print('--- CALENDAR AVAILABILITY AUDIT ---')
print(df_calendar['available'].value_counts(dropna=False))
total_f_days = (df_calendar['available'] == 'f').sum()
total_calendar_rows = len(df_calendar)
print(f'Total Unavailable/Booked (f) Days: {total_f_days:,}')
print(
    f'True Raw Market Occupancy Rate:'
    f' {(total_f_days / total_calendar_rows) * 100:.2f}%'
)

Loading datasets from Google Cloud Storage...
Listings Shape: (500, 11) | Calendar Shape: (182500, 5)
--- LISTINGS PRIMARY KEY AUDIT ---
Total Raw Rows: 500
Unique Listing IDs: 267
Duplicate Listing Rows: 233

Top 5 duplicated listing IDs:
id
1.74E+18    14
9.38E+17    13
1.38E+18     8
1.47E+18     8
1.02E+18     8
Name: count, dtype: int64
--- NULL VALUES IN CORE COLUMNS ---
id                         0
name                       0
neighbourhood_cleansed     0
room_type                  0
accommodates               0
bedrooms                   0
beds                       7
price                     33
review_scores_rating      51
number_of_reviews          0
host_is_superhost          0
dtype: int64
--- CALENDAR AVAILABILITY AUDIT ---
available
t    126152
f     56348
Name: count, dtype: int64
Total Unavailable/Booked (f) Days: 56,348
True Raw Market Occupancy Rate: 30.88%


# Data Cleaning & Transformation

In [ ]:
import pandas_gbq

# Define the query to load the master_analytics table
query = 'SELECT * FROM `airbnb-analysis-513507.airbnb_data.master_analytics`'

# Load the table into a pandas DataFrame
df_master = pandas_gbq.read_gbq(query, project_id='airbnb-analysis-513507')

# Display the first few rows and schema information
print(f'Shape of master_analytics: {df_master.shape}')
display(df_master.head())

Downloading: 100%|██████████|
Shape of master_analytics: (248, 14)


,listing_id,listing_name,neighborhood,room_type,accommodates,bedrooms,beds,price,rating,number_of_reviews,host_status,occupancy_rate,booked_days,estimated_annual_revenue
0,38321579,Comfy and quiet,EIGHTH WARD,Private room,1,1,1,75.00,4.97,507,Superhost,32.60,119,8925.00
1,27692601,Albany Cozy home with fireplaces and Sunroom,EIGHTH WARD,Entire home/apt,4,2,2,344.50,4.86,21,Regular Host,55.89,204,70278.00
2,44450012,Spacious upscale luxury apartment with huge yard,EIGHTH WARD,Entire home/apt,4,2,2,146.54,4.87,30,Superhost,38.08,139,20369.06
3,1.53E+18,Charming 2 BR home in Albany,EIGHTH WARD,Entire home/apt,5,2,2,270.00,4.99,78,Superhost,26.64,389,105030.00
4,1E+18,Elegant Albany Retreat | Mins to D/T & UAlbany,EIGHTH WARD,Entire home/apt,6,3,3,242.50,4.99,113,Superhost,0.00,0,0.00


In [ ]:
# verify the null values
df_master.isna().sum()

,0
listing_id,0
listing_name,0
neighborhood,0
room_type,0
accommodates,0
bedrooms,0
beds,0
price,0
rating,0
number_of_reviews,0


In [ ]:
# inspect the rating
df_master['rating'].value_counts().sort_index(ascending=False)

,count
rating,
5.00,33
4.99,8
4.98,2
4.97,5
4.96,10
4.95,5
4.94,13
4.93,7
4.92,7


In [ ]:
# drop null values from rating column
df_master = df_master.dropna(subset=['rating'])

print("\nNew Shape of df_master",df_master.shape)


New Shape of df_master (248, 14)


In [ ]:
# verify the duplicates
df_master.duplicated().sum()

np.int64(0)

In [ ]:
import pandas_gbq

# Define the target table ID in BigQuery
target_table = 'airbnb-analysis-513507.airbnb_data.master_analytics'

# Write the cleaned DataFrame back to BigQuery (overwriting the old table)
pandas_gbq.to_gbq(
    df_master,
    destination_table=target_table,
    project_id='airbnb-analysis-513507',
    if_exists='replace'
)

print("Cleaned master_analytics table successfully saved back to BigQuery!")

100%|██████████| 1/1 [00:00<00:00, 9664.29it/s]

Cleaned master_analytics table successfully saved back to BigQuery!
